We load the checkerboard images from data/calibration/ and verify that the notebook can access them. The key check is the number of images found (should be 18). If this fails, later steps (corner detection and calibration) cannot run reliably.

In [22]:
import glob
import os
IMAGE_GLOB = "data/calibration/*.jpeg"
images = sorted(glob.glob(IMAGE_GLOB))

print("Folder:", os.path.dirname(IMAGE_GLOB))
print("Found images:", len(images))

# İlk 5 dosyayı gösterelim (kontrol için)
for p in images[:5]:
    print(" -", p)

if len(images) == 0:
    print("\nHATA: Görsel bulunamadı.")
    print("Kontrol et:")
    print(" - Fotoğraflar calib_images/ içinde mi?")
    print(" - Uzantı .jpg mi? Değilse IMAGE_GLOB'u değiştir.")

Folder: data/calibration
Found images: 18
 - data/calibration/calib1.jpeg
 - data/calibration/calib10.jpeg
 - data/calibration/calib11.jpeg
 - data/calibration/calib12.jpeg
 - data/calibration/calib13.jpeg


Step 2 — Detect checkerboard corners (data for calibration)

We detect the inner chessboard corners in each calibration image and keep only the images where detection succeeds. These 2D corner locations (in pixel coordinates) are the measurements used by the calibration algorithm. We also save visual overlays to quickly verify that corner detection is correct (wrong pattern size typically gives many failures or clearly incorrect overlays).

In [23]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install opencv-python numpy matplotlib

In [24]:
import glob, os
import cv2
import numpy as np

IMAGE_GLOB = "data/calibration/*.jpeg"
images = sorted(glob.glob(IMAGE_GLOB))
assert len(images) > 0

pattern_size = (9, 6)  # (cols, rows) inner corners
out_dir = "out_corners"
os.makedirs(out_dir, exist_ok=True)

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3)

imgpoints = []
valid_paths = []

ok = 0
for i, path in enumerate(images):
    img = cv2.imread(path)
    assert img is not None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    found, corners = cv2.findChessboardCorners(
        gray,
        pattern_size,
        flags=cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE
    )

    if not found:
        continue

    corners = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)

    imgpoints.append(corners)
    valid_paths.append(path)
    ok += 1

    vis = img.copy()
    cv2.drawChessboardCorners(vis, pattern_size, corners, found)
    cv2.imwrite(os.path.join(out_dir, f"corners_{i:02d}.jpg"), vis)

print("Detected corners in:", ok, "/", len(images))
print("Saved overlays to:", out_dir)

Detected corners in: 18 / 18
Saved overlays to: out_corners


Using the detected 2D corners and the known 3D checkerboard grid, we estimate the camera intrinsics K and the lens distortion coefficients dist with cv2.calibrateCamera. We also compute the mean reprojection error to assess calibration quality (lower is better).

In [ ]:
import cv2
import numpy as np

assert len(imgpoints) == len(valid_paths) and len(imgpoints) > 0

square_size = 0.02
cols, rows = pattern_size
objp = np.zeros((rows * cols, 3), np.float32)
objp[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2)
objp *= square_size

objpoints = [objp.copy() for _ in range(len(imgpoints))]

sample = cv2.imread(valid_paths[0])
h, w = sample.shape[:2]

ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, (w, h), None, None
)

total_err = 0.0
total_points = 0

for i in range(len(objpoints)):
    proj, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], K, dist)
    err = cv2.norm(imgpoints[i], proj, cv2.NORM_L2)
    n = len(proj)
    total_err += err * err
    total_points += n

rmse = np.sqrt(total_err / total_points)

print("RMS (OpenCV):", ret)
print("K:\n", K)
print("dist:\n", dist)
print("Reprojection RMSE (px):", rmse)

RMS (OpenCV): 0.6038950516777517
K:
 [[1.47014833e+03 0.00000000e+00 7.68325185e+02]
 [0.00000000e+00 1.47057687e+03 1.03334756e+03]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
dist:
 [[ 2.01895611e-01 -7.47038699e-01  4.35410824e-04  1.28308511e-03
   7.88245845e-01]]
Reprojection RMSE (px): 0.6038951801016637


Step 4 — Undistort a few images (sanity check)

We use the estimated K and dist to undistort sample calibration images. This is a quick visual sanity check that lens distortion is being corrected (straight lines should look straighter, and the board should look less “bulged”).

In [26]:
import os, cv2
import numpy as np

out_dir = "out_undistort"
os.makedirs(out_dir, exist_ok=True)

for i, path in enumerate(valid_paths[:5]):
    img = cv2.imread(path)
    h, w = img.shape[:2]

    newK, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), 1.0, (w, h))
    und = cv2.undistort(img, K, dist, None, newK)

    out_path = os.path.join(out_dir, f"undist_{i:02d}.jpg")
    cv2.imwrite(out_path, und)

print("Saved undistorted samples to:", out_dir)

Saved undistorted samples to: out_undistort


Step 5 — Save calibration (K, dist) for later use

We store the estimated intrinsics K and distortion dist to disk so the same parameters can be reused in later sections (Essential matrix estimation, epipolar geometry, triangulation) without recalibrating.

In [27]:
import numpy as np

np.savez("calibration_results.npz", K=K, dist=dist, pattern_size=np.array(pattern_size), rmse=np.array(rmse))

K_inv = np.linalg.inv(K)
print("Saved: calibration_results.npz")
print("K_inv:\n", K_inv)

Saved: calibration_results.npz
K_inv:
 [[ 6.80203472e-04  0.00000000e+00 -5.22617458e-01]
 [ 0.00000000e+00  6.80005257e-04 -7.02681773e-01]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
